## Widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "project_dev")
dbutils.widgets.text("storageName", "adlssmartprojectdev13")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")

# definir rutas
path_destino = f"abfss://{container}@{storageName}.dfs.core.windows.net/raw_data_project/"
path_volume = f"/Volumes/{catalogo}/{container}/datasets/"

In [0]:
%sql
USE CATALOG ${catalogo};

In [0]:
print("Username:", dbutils.secrets.get("ScopeforKaggle", "usernamedbtKag"))

In [0]:
import os

import json
from kaggle.api.kaggle_api_extended import KaggleApi

# Crear carpeta de configuración
kaggle_dir = "/root/.kaggle"
os.makedirs(kaggle_dir, exist_ok=True)

# Crear archivo kaggle.json con secrets
with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({
        "username": dbutils.secrets.get("ScopeforKaggle", "usernamedbtKag"),
        "key": dbutils.secrets.get("ScopeforKaggle", "keydbtKag")
    }, f)

# Permisos requeridos por Kaggle
os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)

# Autenticación
api = KaggleApi()
api.authenticate()

print("✅ Autenticado correctamente en Kaggle")

## Descargar archivos en el volume

In [0]:
'''import os
import certifi
from kaggle.api.kaggle_api_extended import KaggleApi

# 🔐 Credenciales desde secrets
os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get("ScopeforKaggle", "usernamedbtKag")
os.environ['KAGGLE_KEY'] = dbutils.secrets.get("ScopeforKaggle", "keydbtKag")

api = KaggleApi()
api.authenticate()'''

# 2. Definir ruta local temporal (en el Driver) y ruta final (ADLS)
print(f"🚀 Descargando directamente al Volumen: {path_volume}")
api.dataset_download_files(
    'olistbr/brazilian-ecommerce', 
    path=path_volume, 
    unzip=True
)

print("✅ Archivos disponibles en el Volumen.")



##Mover Archivos azure datalake

In [0]:
## eliminar archivos en el container
dbutils.fs.rm( path_destino, recurse=True )

In [0]:
# copiar archivos en datalake
dbutils.fs.cp(
    path_volume,
    path_destino,
    recurse=True
)

In [0]:
## ELIMINAR ARCHIVOS VOLUME
dbutils.fs.rm(path_volume, True)